# Python 面向对象进阶：封装、继承、多态与图书管理系统

> 建议课时：50–60 分钟  
> 适合对象：已经学习过 Python `class`、对象、属性、方法、`__init__`、`self` 的同学。

---

## 本节课目标

学完本节课后，你应该能够：

1. 理解什么是**封装**，知道为什么不要随意直接修改对象内部数据。
2. 理解什么是**继承**，知道如何通过子类复用父类代码。
3. 理解什么是**方法重写**，知道子类如何改写父类方法。
4. 理解什么是**多态**，知道同一个方法在不同对象上可以有不同表现。
5. 使用多个类协作完成一个简单项目：**图书管理系统**。

---

# 1. 复习：类和对象

上一节课我们学习了如何定义一个类。

类可以理解成一个“模板”，对象是根据这个模板创建出来的具体实例。

例如：

- `Student` 是类。
- `s1 = Student("Tom", 90)` 创建出来的 `s1` 是对象。
- `name`、`score` 是对象的属性。
- `show_info()` 是对象的方法。

In [1]:
class Student:
    def __init__(self, name: str, score: int):
        self.name = name
        self.score = score

    def show_info(self) -> None:
        print(f"学生姓名：{self.name}，成绩：{self.score}")


s1 = Student("Tom", 90)
s2 = Student("Alice", 85)

s1.show_info()
s2.show_info()

学生姓名：Tom，成绩：90
学生姓名：Alice，成绩：85


## 思考问题

如果我们只会写一个简单的类，通常还不够。

真实项目里，我们经常会遇到这些问题：

1. 对象里的数据能不能随便改？
2. 两个类有很多重复代码，应该怎么办？
3. 不同对象都有同名方法，但表现不一样，应该怎么设计？
4. 多个类之间如何配合完成一个功能？

这些问题就对应本节课的三个核心概念：

- 封装
- 继承
- 多态

# 2. 封装：把数据和操作放在一起

封装的核心思想是：

> 把数据和操作这些数据的方法放在同一个类里，并尽量避免外部随意修改对象内部状态。

先看一个银行账户类。

In [ ]:
class BankAccount:
    def __init__(self, owner: str, balance: float):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:
        self.balance += amount

    def withdraw(self, amount: float) -> bool:
        if amount > self.balance:
            print("余额不足，取款失败")
            return False
        self.balance -= amount
        print("取款成功")
        return True

    def show_balance(self) -> None:
        print(f"{self.owner} 当前余额：{self.balance}")


account = BankAccount("Tom", 1000)
account.show_balance() # print(account.balance)

account.deposit(500) # account.balance += 500
account.show_balance() # print(account.balance)

account.withdraw(300) # account.balance -= 300
account.show_balance() # print(account.balance)



Tom 当前余额：1000
Tom 当前余额：1500
取款成功
Tom 当前余额：1200


这个类看起来没问题，但有一个风险：外部可以直接修改 `balance`。

In [ ]:
account = BankAccount("Tom", 1000)
account.balance = -9999
account.show_balance()

余额被改成负数了，这在真实业务里通常是不合理的。

所以我们希望：

- 外部不要直接修改余额。
- 修改余额必须通过 `deposit()` 和 `withdraw()` 方法。

在 Python 里，常见做法是把内部属性写成 `_balance`。

注意：单下划线不是强制私有，只是一种约定，表示“这个属性主要供类内部使用，外部不建议直接访问”。

In [ ]:
class SafeBankAccount:
    def __init__(self, owner: str, balance: float):
        self.owner = owner
        self._balance = balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            print("存款金额必须大于 0")
            return
        self._balance += amount

    def withdraw(self, amount: float) -> bool:
        if amount <= 0:
            print("取款金额必须大于 0")
            return False
        if amount > self._balance:
            print("余额不足，取款失败")
            return False
        self._balance -= amount
        print("取款成功")
        return True

    def get_balance(self) -> float:
        return self._balance

    def show_balance(self) -> None:
        print(f"{self.owner} 当前余额：{self._balance}")


account = SafeBankAccount("Alice", 1000)
account.deposit(200)
account.withdraw(500)
account.show_balance()
print("通过方法获取余额：", account.get_balance())

## 封装小结

封装的好处：

1. 数据和操作放在一起，代码更清晰。
2. 外部不直接修改内部数据，减少错误。
3. 修改内部实现时，外部调用方式可以保持不变。

常见写法：

```python
self._xxx
```

表示这是内部属性，不建议外部直接访问。

# 3. 继承：复用已有类

继承的核心思想是：

> 子类可以复用父类已有的属性和方法，也可以新增自己的属性和方法。

例如：

- `Employee` 表示普通员工。
- `Manager` 表示经理。

经理也是员工，所以 `Manager` 可以继承 `Employee`。

In [3]:
class Employee:
    def __init__(self, name: str, salary: float):
        self.name = name
        self.salary = salary

    def show_info(self) -> None:
        print(f"员工：{self.name}，薪资：{self.salary}")


class Manager(Employee):
    def __init__(self, name: str, salary: float, department: str):
        super().__init__(name, salary)
        self.department = department

    def show_info(self) -> None:
        print(f"经理：{self.name}，部门：{self.department}，薪资：{self.salary}")


emp = Employee("Tom", 8000)
mgr = Manager("Alice", 15000, "技术部")

emp.show_info()
mgr.show_info()

员工：Tom，薪资：8000
经理：Alice，部门：技术部，薪资：15000


## 代码解释

```python
class Manager(Employee):
```

表示 `Manager` 继承 `Employee`。

```python
super().__init__(name, salary)
```

表示调用父类 `Employee` 的初始化方法，复用父类已有的 `name` 和 `salary` 属性。

```python
self.department = department
```

表示子类新增自己的属性。

```python
def show_info(self):
```

子类重新定义了和父类同名的方法，这叫**方法重写**。

## 继承的适用场景

当两个类之间存在“是一个”的关系时，可以考虑继承。

例如：

- 经理 是一个 员工
- 狗 是一个 动物
- 猫 是一个 动物
- 电动车 是一个 交通工具

但是不要滥用继承。

如果两个类只是“拥有”的关系，通常不适合继承。

例如：

- 学生 拥有 书包
- 图书馆 拥有 很多书
- 订单 拥有 很多商品

这种更适合用“组合”，也就是把一个对象作为另一个对象的属性。

# 4. 方法重写：子类改变父类行为

子类可以继承父类方法，也可以重写父类方法。

下面看一个动物类案例。

In [4]:
class Animal:
    def __init__(self, name: str):
        self.name = name

    def speak(self) -> None:
        print(f"{self.name} 发出声音")


class Dog(Animal):
    def speak(self) -> None:
        print(f"{self.name}：汪汪！")


class Cat(Animal):
    def speak(self) -> None:
        print(f"{self.name}：喵喵！")


animal = Animal("普通动物")
dog = Dog("小黑")
cat = Cat("小白")

animal.speak()
dog.speak()
cat.speak()

普通动物 发出声音
小黑：汪汪！
小白：喵喵！


`Dog` 和 `Cat` 都继承自 `Animal`，但是它们都有自己的 `speak()` 实现。

这就是方法重写。

# 5. 多态：同一个方法，不同表现

多态的核心思想是：

> 不同类型的对象，可以使用相同的方法名，但表现出不同的行为。

继续使用动物案例。

In [5]:
animals = [
    Dog("旺财"),
    Cat("咪咪"),
    Animal("未知动物")
]

for animal in animals:
    animal.speak()

旺财：汪汪！
咪咪：喵喵！
未知动物 发出声音


这里虽然循环里统一调用：

```python
animal.speak()
```

但是不同对象会执行自己的 `speak()` 方法。

- Dog 对象输出“汪汪”
- Cat 对象输出“喵喵”
- Animal 对象输出“发出声音”

这就是多态。

多态可以让代码更灵活，减少大量 `if...else...` 判断。

# 6. 项目案例：图书管理系统

接下来我们用多个类协作完成一个简单项目。

项目需求：

1. 可以创建图书。
2. 可以把图书加入图书馆。
3. 可以展示所有图书。
4. 可以借书。
5. 可以还书。

我们设计两个类：

| 类名 | 作用 |
|---|---|
| `Book` | 表示一本书 |
| `Library` | 表示一个图书馆，管理多本书 |

这里体现的是“组合”关系：

> 图书馆拥有很多书。

## 6.1 定义 Book 类

一本书应该有：

- 书名 `title`
- 作者 `author`
- 是否已借出 `is_borrowed`

同时我们定义 `__str__()`，让打印图书对象时更友好。

In [6]:
class Book:
    def __init__(self, title: str, author: str, is_borrowed: bool = False):
        self.title = title
        self.author = author
        self.is_borrowed = is_borrowed

    def __str__(self) -> str:
        status = "已借出" if self.is_borrowed else "可借阅"
        return f"《{self.title}》 - {self.author} - {status}"


book = Book("Python基础", "张三")
print(book)

《Python基础》 - 张三 - 可借阅


## 6.2 定义 Library 类

图书馆需要维护一个图书列表：

```python
self.books: list[Book] = []
```

它需要提供这些方法：

- `add_book()`：添加图书
- `show_books()`：展示所有图书
- `borrow_book()`：借书
- `return_book()`：还书

In [ ]:
class Library:
    def __init__(self):
        self.books: list[Book] = []

    def add_book(self, book: Book) -> None:
        self.books.append(book)
        print(f"已添加图书：{book.title}")

    def show_books(self) -> None:
        if not self.books:
            print("图书馆暂无图书")
            return

        print("当前图书列表：")
        for book in self.books:
            print(book)

    def borrow_book(self, title: str) -> bool:
        for book in self.books:
            if book.title == title:
                if book.is_borrowed:
                    print(f"《{title}》已经被借出")
                    return False
                book.is_borrowed = True
                print(f"成功借出《{title}》")
                return True

        print(f"没有找到《{title}》")
        return False

    def return_book(self, title: str) -> bool:
        for book in self.books:
            if book.title == title:
                if not book.is_borrowed:
                    print(f"《{title}》未被借出，不需要归还")
                    return False
                book.is_borrowed = False
                print(f"成功归还《{title}》")
                return True

        print(f"没有找到《{title}》")
        return False

## 6.3 使用图书管理系统

In [ ]:
library = Library()

library.add_book(Book("Python基础", "张三"))
library.add_book(Book("数据分析入门", "李四"))
library.add_book(Book("机器学习实战", "王五"))

print()
library.show_books()

print("\n借书操作：")
library.borrow_book("Python基础")

print("\n借书后：")
library.show_books()

print("\n还书操作：")
library.return_book("Python基础")

print("\n还书后：")
library.show_books()

## 6.4 项目案例总结

这个图书管理系统使用到了：

1. `Book` 类：表示一本书。
2. `Library` 类：表示图书馆。
3. 对象之间的组合：图书馆中保存多个图书对象。
4. 类型标注：`list[Book]`、`book: Book`、`title: str`。
5. 方法封装：借书、还书、展示图书都封装成方法。

这比把所有变量和函数都散落在外面更清晰。

# 7. 课堂练习 1：员工类和经理类

## 要求

请完成下面任务：

1. 定义 `Employee` 类。
2. `Employee` 包含 `name` 和 `salary` 两个属性。
3. `Employee` 有 `show_info()` 方法，用来打印员工信息。
4. 定义 `Manager` 类，继承 `Employee`。
5. `Manager` 增加 `department` 属性。
6. `Manager` 重写 `show_info()` 方法，打印经理信息。

## 预期效果

```python
emp = Employee("Tom", 8000)
mgr = Manager("Alice", 15000, "技术部")

emp.show_info()
mgr.show_info()
```

输出类似：

```text
员工：Tom，薪资：8000
经理：Alice，部门：技术部，薪资：15000
```

In [ ]:
# 课堂练习 1：请在这里完成代码

## 课堂练习 1 参考答案

In [ ]:
class Employee:
    def __init__(self, name: str, salary: float):
        self.name = name
        self.salary = salary

    def show_info(self) -> None:
        print(f"员工：{self.name}，薪资：{self.salary}")


class Manager(Employee):
    def __init__(self, name: str, salary: float, department: str):
        super().__init__(name, salary)
        self.department = department

    def show_info(self) -> None:
        print(f"经理：{self.name}，部门：{self.department}，薪资：{self.salary}")


emp = Employee("Tom", 8000)
mgr = Manager("Alice", 15000, "技术部")

emp.show_info()
mgr.show_info()

# 8. 课堂练习 2：动物叫声多态

## 要求

请完成下面任务：

1. 定义 `Animal` 类。
2. `Animal` 有 `speak()` 方法。
3. 定义 `Dog` 类，继承 `Animal`，重写 `speak()`。
4. 定义 `Cat` 类，继承 `Animal`，重写 `speak()`。
5. 创建一个列表，里面放入 `Dog` 和 `Cat` 对象。
6. 使用循环调用每个对象的 `speak()` 方法。

## 预期效果

```text
狗：汪汪
猫：喵喵
狗：汪汪
```

In [ ]:
# 课堂练习 2：请在这里完成代码

## 课堂练习 2 参考答案

In [ ]:
class Animal:
    def speak(self) -> None:
        print("动物发出声音")


class Dog(Animal):
    def speak(self) -> None:
        print("狗：汪汪")


class Cat(Animal):
    def speak(self) -> None:
        print("猫：喵喵")


animals = [Dog(), Cat(), Dog()]

for animal in animals:
    animal.speak()

# 9. 本节课总结

本节课我们学习了 Python 面向对象的三个进阶概念。

## 1. 封装

把数据和操作数据的方法放在一起。

```python
class BankAccount:
    def __init__(self, balance: float):
        self._balance = balance
```

`_balance` 表示内部属性，不建议外部直接修改。

---

## 2. 继承

子类可以复用父类已有代码。

```python
class Manager(Employee):
    pass
```

---

## 3. 方法重写

子类可以定义和父类同名的方法，改变行为。

```python
class Dog(Animal):
    def speak(self):
        print("汪汪")
```

---

## 4. 多态

同一个方法名，不同对象有不同表现。

```python
for animal in animals:
    animal.speak()
```

---

## 5. 多个类协作

真实项目里，经常不是一个类单独工作，而是多个类一起完成任务。

例如：

- `Book` 表示一本书。
- `Library` 管理多本书。

# 10. 课后作业：电影票预订系统

## 作业目标

请使用面向对象编程完成一个简单的电影票预订系统。

需要设计两个类：

| 类名 | 作用 |
|---|---|
| `Movie` | 表示一部电影 |
| `Cinema` | 表示电影院，管理多部电影 |

---

## Movie 类要求

`Movie` 类需要包含以下属性：

1. `title: str`：电影名称
2. `duration: int`：电影时长，单位分钟
3. `seats: int`：剩余座位数

建议添加 `__str__()` 方法，让打印电影信息更方便。

---

## Cinema 类要求

`Cinema` 类需要包含：

1. `movies: list[Movie]`：电影列表
2. `add_movie(movie: Movie) -> None`：添加电影
3. `show_movies() -> None`：展示所有电影
4. `book_ticket(title: str, count: int) -> bool`：预订电影票

---

## 功能要求

1. 可以添加电影。
2. 可以展示所有电影。
3. 可以根据电影名预订电影票。
4. 如果电影不存在，提示“电影不存在”。
5. 如果座位不足，提示“座位不足”。
6. 如果预订成功，减少对应电影的剩余座位数。
7. 代码需要使用 Python3 类型标注。

---

## 测试示例

```python
cinema = Cinema()

cinema.add_movie(Movie("流浪地球", 125, 50))
cinema.add_movie(Movie("哪吒", 110, 30))

cinema.show_movies()

cinema.book_ticket("哪吒", 2)
cinema.book_ticket("流浪地球", 60)
cinema.book_ticket("不存在的电影", 1)

cinema.show_movies()
```

---

## 提示

可以参考本节课的图书管理系统：

- `Book` 类对应 `Movie` 类。
- `Library` 类对应 `Cinema` 类。
- `borrow_book()` 对应 `book_ticket()`。

In [ ]:
# 课后作业：请在这里完成电影票预订系统
class Movie:
    def __init__(self, title: str, time: int, seat: int) -> None:
        self.title = title
        self.time = time
        self.seat = seat

    def show_info(self) -> None:
      
        print(f"《{self.title}》，时长：{self.time},剩余座位：{self.seat}")

    def __str__(self) -> str:
        
        if self.seat <= 0:
            status = "无空位"
        else:
            status = "有空位"
            self.seat -= 1
            
        return f"《{self.title}》 - {self.time} - {status}"


class Cinema:
    def __init__(self):
        self.movies: list[Movie] = []

    def add_movies(self, movie: Movie) -> None:
        self.movies.append(movie)
        print(f"已添加电影（{movie.title}）")

    def show_movies(self) -> None:
        if not self.movies:
            print("目前场内无电影")
            return
        print("当前图书列表：")
        for movie in self.movies:
            print(movie)

    def take_ticket(self, title: str,count: int) -> bool:
        self.count = count
        for movie in self.movies:
            if movie.title == title:
                movie.seat -= self.count
                if movie.seat <= 0:
                    print(f"{title}已满员，无法入席")
                    return False  
                else:
                    print(f"成功预约{title}，剩余{movie.seat}")
                    return True
        print(f"不存在的电影")
        return False



cinema = Cinema()
cinema.add_movies(Movie("流浪地球", 125, 50))
cinema.add_movies(Movie("哪吒", 110, 30))

cinema.show_movies()

cinema.take_ticket("哪吒", 2)
cinema.take_ticket("流浪地球", 60)
cinema.take_ticket("不存在的电影", 1)

cinema.show_movies()
